# Clasification 

Customers about to leave and wanna know the probability of the risk to leave of each of them

## Binnary Classification - Data Prep



In [76]:
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 


In [77]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

!wget $data -O data-week-3.csv

--2026-10-09 06:47:57--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  --.-KB/s    in 0.003s  

2026-10-09 06:47:57 (306 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [78]:
#PATH = 'Lesson3-Classification'
df = pd.read_csv('data-week-3.csv')
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [79]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [80]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [81]:
df.dtypes


customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [82]:
# pd.to_numeric(df.totalcharges)


In [83]:
tc = pd.to_numeric(df.totalcharges, errors='coerce')


In [84]:
tc.isnull().sum()

np.int64(11)

In [85]:
df[tc.isnull()][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-LVYGI,
753,3115-CZMZD,
936,5709-LVOEQ,
1082,4367-NUYAO,
1340,1371-DWPAZ,
3331,7644-OMVMY,
3826,3213-VVOLG,
4380,2520-SGTTA,
5218,2923-ARZLG,
6670,4075-WKNIU,


In [86]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

In [87]:
df[tc.isnull()][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-LVYGI,0.0
753,3115-CZMZD,0.0
936,5709-LVOEQ,0.0
1082,4367-NUYAO,0.0
1340,1371-DWPAZ,0.0
3331,7644-OMVMY,0.0
3826,3213-VVOLG,0.0
4380,2520-SGTTA,0.0
5218,2923-ARZLG,0.0
6670,4075-WKNIU,0.0


# Encoding the Variable

In [88]:
df.churn.head()

0     No
1     No
2    Yes
3     No
4    Yes
Name: churn, dtype: str

We need numbers for classification, this produces a boolean series casting them into integers

In [89]:
df.churn = df.churn.str.lower().eq('yes').astype(int)

In [90]:
df.churn.head()


0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

# 03 - Validation

We need to split the dataset from training to validation set  60% - training 20% validation and 20% testing 

This is section says how to isolates out target inside the dataste wich are the churn values

In [91]:
from sklearn.model_selection import train_test_split

Making aside 20 percent of the data

In [92]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [93]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [94]:
len(df_train), len(df_val), len(df_test)

(4225, 1409, 1409)

In [95]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

Here we need to isolate the target we are extracting it from each dataframe as numpy array

In [96]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

In [97]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

# 04 eda

In [98]:
df_full_train = df_full_train.reset_index(drop=True)

In [99]:
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [100]:
df_full_train.churn.value_counts(normalize=True).rename_axis(None).rename('churn')

0    0.730032
1    0.269968
Name: churn, dtype: float64

In [101]:
df_full_train.churn.mean()

np.float64(0.26996805111821087)

In [102]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

In [103]:
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [104]:
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

The EDA for this project consisted of:

Checking missing values
Looking at the distribution of the target variable (churn)
Looking at numerical and categorical variables
Functions and methods:

df.isnull().sum() - returns the number of null values in the dataframe.
df.x.value_counts() returns the number of values for each category in x series. The normalize=True argument retrieves the percentage of each category. In this project, the mean of churn is equal to the churn rate obtained with the value_counts method.
round(x, y) - round an x number with y decimal places
df[x].nunique() - returns the number of unique values in x series

El error estaba en la celda que convertía churn a números:

df.churn = (df.churn == 'yes').astype(int)
 
Los valores originales eran "Yes" y "No" (con mayúscula), así que la comparación con "yes" nunca coincidía. Por eso todos quedaban como 0.

La cambié para ignorar mayúsculas y minúsculas:

df.churn = df.churn.str.lower().eq('yes').astype(int)



# 05 Risk